# Introducción a Tool Calling

Este notebook recorre el protocolo de tool calling con un modelo local (Ollama): definición de herramientas con Pydantic, `bind_tools()`, dispatcher, llamadas paralelas, reintentos y un ciclo completo.

Ejecute cada celda en orden. Requiere Ollama con el modelo `qwen2.5:3b`.

## Definición de herramientas con Pydantic

In [1]:
from langchain_ollama import ChatOllama
# temperature=0.0 enforces deterministic output — critical for tool calling,
# where a non-zero temperature can cause the model to skip tools, invent
# argument values, or answer directly when a tool call is expected.
language_model = ChatOllama(model = "qwen2.5:3b", temperature = 0.0)

In [2]:
from pydantic import BaseModel, Field
from langchain_core.tools import tool
import operator
import json
import ast

### Herramienta de conversión de unidades

In [3]:
# Each key is a (from_unit, to_unit) tuple; the value is a conversion lambda.
# Lookup is O(1) and adding a new unit pair requires only a new entry here —
# no changes to the tool logic are needed.
CONVERSIONS = {
    ("km", "miles"):             lambda x: x * 0.621371,
    ("miles", "km"):             lambda x: x * 1.60934,
    ("celsius", "fahrenheit"):   lambda x: x * 9 / 5 + 32,
    ("fahrenheit", "celsius"):   lambda x: (x - 32) * 5 / 9,
    ("kg", "lbs"):               lambda x: x * 2.20462,
    ("lbs", "kg"):               lambda x: x * 0.453592,
    ("meters", "feet"):          lambda x: x * 3.28084,
    ("feet", "meters"):          lambda x: x * 0.3048,
    ("liters", "gallons"):       lambda x: x * 0.264172,
    ("gallons", "liters"):       lambda x: x * 3.78541,
}

In [4]:
class ConvertUnitsInput(BaseModel):

    """Input schema for the convert_units tool.

    The Field descriptions are the primary guidance the LLM uses when choosing
    argument values. Listing the supported units explicitly steers the model
    toward valid inputs and reduces argument-mismatch errors at runtime.
    """

    value: float = Field(description = "The numeric value to convert")
    from_unit: str = Field(
        description = "Source unit: km, miles, celsius, fahrenheit, kg, lbs, meters, feet, liters, gallons"
    )
    to_unit: str = Field(
        description = "Target unit: km, miles, celsius, fahrenheit, kg, lbs, meters, feet, liters, gallons"
    )

In [5]:
@tool("convert_units", args_schema = ConvertUnitsInput)
def convert_units(value: float, from_unit: str, to_unit: str) -> str:

    """Convert a numeric value from one unit of measurement to another.

    Supports ten bidirectional pairs: km↔miles, celsius↔fahrenheit, kg↔lbs,
    meters↔feet, liters↔gallons. Returns a formatted equality string on success
    or a descriptive message listing valid pairs when the combination is not
    supported — this lets the LLM relay the error to the user rather than
    presenting an opaque failure.

    Args:
        value:     The numeric quantity to convert.
        from_unit: The source unit name (case-insensitive, whitespace stripped).
        to_unit:   The target unit name (same normalisation as from_unit).

    Returns:
        "{value} {from_unit} = {result:.4f} {to_unit}" on success, or an
        unsupported-pair message if the combination is not in CONVERSIONS.
    """

    # Normalise before lookup so "KM", " km " and "km" all resolve correctly.
    key = (from_unit.lower().strip(), to_unit.lower().strip())
    if key in CONVERSIONS:
        result = CONVERSIONS[key](value)
        return f"{value} {from_unit} = {result:.4f} {to_unit}"

    return (
        f"Conversion from '{from_unit}' to '{to_unit}' is not supported. "
        f"Supported pairs: km/miles, celsius/fahrenheit, kg/lbs, meters/feet, liters/gallons."
    )

print(f"TOOL DEFINED = {convert_units.name}")

TOOL DEFINED = convert_units


### Herramienta de cálculo aritmético seguro

In [6]:
# Whitelist of AST node types mapped to their safe Python operator equivalents.
# Any node NOT listed here — function calls, attribute access, subscripts,
# variable names, imports — causes _safe_eval to raise ValueError, preventing
# code injection regardless of what string the LLM generates.
ALLOWED_OPS = {
    ast.Add:  operator.add,
    ast.Sub:  operator.sub,
    ast.Mult: operator.mul,
    ast.Div:  operator.truediv,
    ast.Pow:  operator.pow,
    ast.Mod:  operator.mod,
    ast.USub: operator.neg,
}

In [7]:
def _safe_eval(node):

    """Recursively evaluate an AST expression node using only allowed arithmetic.

    Walks the abstract syntax tree produced by ast.parse() and applies the
    corresponding Python operator for each node type. The function is
    intentionally restrictive: it accepts only numeric constants, binary
    operations (e.g. +, *, **), and unary negation. Any other node type —
    including function calls and name lookups — raises ValueError, blocking
    code injection at the AST level before any execution occurs.

    Args:
        node: An ast.expr node, typically tree.body from
              ast.parse(expression, mode="eval").

    Returns:
        The numeric result (int or float) of the expression.

    Raises:
        ValueError: If the node type is not in ALLOWED_OPS or is not a numeric
                    constant. The caller wraps this in a user-facing message.
    """

    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value

    elif isinstance(node, ast.BinOp) and type(node.op) in ALLOWED_OPS:
        return ALLOWED_OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))

    elif isinstance(node, ast.UnaryOp) and type(node.op) in ALLOWED_OPS:
        return ALLOWED_OPS[type(node.op)](_safe_eval(node.operand))

    # Anything else — Call, Attribute, Name, Import — is rejected.
    raise ValueError(f"Unsupported expression type: {type(node).__name__}")

In [8]:
class CalculateInput(BaseModel):

    """Input schema for the calculate tool.

    The description lists supported operators explicitly so the LLM knows what
    syntax is valid. Expressions with function calls (sqrt, abs) or variable
    names are not supported and will be rejected by _safe_eval.
    """

    expression: str = Field(
        description = "A mathematical expression using +, -, *, /, **, % operators and numbers"
    )

In [9]:
@tool("calculate", args_schema = CalculateInput)
def calculate(expression: str) -> str:

    """Evaluate a mathematical expression safely without using eval().

    Uses ast.parse() to convert the expression string into a syntax tree, then
    walks it with _safe_eval(), which only executes nodes in the ALLOWED_OPS
    whitelist. This prevents code injection: an expression like
    __import__('os').system('rm -rf /') parses to a Call node, which _safe_eval
    rejects with ValueError before any code runs.

    Args:
        expression: A string containing a mathematical expression, e.g. "2 ** 10"
                    or "15 * 7 + 3". Supports +, -, *, /, **, % and unary minus.

    Returns:
        A string of the form "{expression} = {result}".

    Raises:
        ValueError: If the expression contains unsupported syntax or node types.
    """

    try:
        tree = ast.parse(expression, mode = "eval")
        result = _safe_eval(tree.body)
        return f"{expression} = {result}"

    except ValueError as e:
        raise ValueError(f"Invalid expression '{expression}': {e}")

    except SyntaxError:
        raise ValueError(f"Syntax error in expression: '{expression}'")


print(f"TOOL DEFINED = {calculate.name}")

TOOL DEFINED = calculate


### Inspección del esquema JSON

In [10]:
# LangChain serialises each Pydantic model to JSON Schema and includes it in
# every call to the LLM. Printing it here shows exactly what the model receives
# when deciding whether and how to invoke convert_units.
print("========== 'convert_units' JSON SCHEMA ==========\n")

schema = convert_units.args_schema.model_json_schema()
print(json.dumps(schema, indent = 2))

========== 'convert_units' JSON SCHEMA ==========

{
  "description": "Input schema for the convert_units tool.\n\nThe Field descriptions are the primary guidance the LLM uses when choosing\nargument values. Listing the supported units explicitly steers the model\ntoward valid inputs and reduces argument-mismatch errors at runtime.",
  "properties": {
    "value": {
      "description": "The numeric value to convert",
      "title": "Value",
      "type": "number"
    },
    "from_unit": {
      "description": "Source unit: km, miles, celsius, fahrenheit, kg, lbs, meters, feet, liters, gallons",
      "title": "From Unit",
      "type": "string"
    },
    "to_unit": {
      "description": "Target unit: km, miles, celsius, fahrenheit, kg, lbs, meters, feet, liters, gallons",
      "title": "To Unit",
      "type": "string"
    }
  },
  "required": [
    "value",
    "from_unit",
    "to_unit"
  ],
  "title": "ConvertUnitsInput",
  "type": "object"
}


## Vinculación de herramientas al modelo

In [11]:
from langchain_core.messages import HumanMessage
# bind_tools() attaches the JSON Schema of each tool to the model so it is
# included in every subsequent invoke() call. The returned object has the same
# interface as language_model but the model can now respond with structured
# tool_calls instead of — or in addition to — plain text.
tools = [convert_units, calculate]
llm_with_tools = language_model.bind_tools(tools)

print(f"MODEL BINDED WITH {len(tools)} TOOL(S): {[t.name for t in tools]}")

MODEL BINDED WITH 2 TOOL(S): ['convert_units', 'calculate']


### Primera invocación

In [12]:
# When the model decides to use a tool, response.content is an empty string
# and response.tool_calls holds a list of structured call objects. When it
# answers directly, response.content has the text and tool_calls is empty.
response = llm_with_tools.invoke([HumanMessage(content = "Convert 100 km to miles")])

print(f"CONTENT = {repr(response.content)}")
print(f"TOOL CALLS = {response.tool_calls}")

CONTENT = ''
TOOL CALLS = [{'name': 'convert_units', 'args': {'value': 100, 'from_unit': 'km', 'to_unit': 'miles'}, 'id': '4fd5cc41-1536-4e5f-abf8-bdfa2d2ddd1e', 'type': 'tool_call'}]


## Ejecución de herramientas

In [13]:
from langchain_core.messages import ToolMessage
# Index tools by name for O(1) dispatch in execute_tool_calls. Using the
# tool's .name attribute (set by @tool) guarantees the key matches the name
# the model sends in each tool_call dict.
TOOLS_MAP = {t.name: t for t in tools}

In [14]:
def execute_tool_calls(tool_calls: list) -> list:

    """Execute every tool call in the list and return one ToolMessage per result.

    Iterates over the tool_calls from the model response, looks each tool up by
    name in TOOLS_MAP, invokes it with the provided arguments, and wraps the
    result in a ToolMessage. The tool_call_id field of each ToolMessage must
    match the id of the corresponding tool_call — the model uses this pairing
    to associate each result with its original request.

    Errors are caught per-call and converted to error strings inside the
    ToolMessage rather than propagating as exceptions. This keeps the agent
    loop alive when one tool fails, letting the model acknowledge the failure
    and attempt a different approach.

    Args:
        tool_calls: List of tool_call dicts from response.tool_calls. Each dict
                    contains 'name', 'args', and 'id'.

    Returns:
        List of ToolMessage objects, one per tool_call, in the same order.
    """

    results = []
    for call in tool_calls:
        tool_fn = TOOLS_MAP.get(call["name"])

        # Unknown tool name: return an error message so the model can react.
        if tool_fn is None:
            content = f"Error: unknown tool '{call['name']}'"
        else:
            try:
                content = str(tool_fn.invoke(call["args"]))
            except Exception as e:
                content = f"Error executing '{call['name']}': {e}"

        results.append(ToolMessage(content = content, tool_call_id = call["id"]))
        print(f"  [{call['name']}] → {content}")

    return results


print("DISPATCHER DEFINED")

DISPATCHER DEFINED


### Ciclo completo de una herramienta

In [15]:
messages = [HumanMessage(content = "Convert 100 km to miles")]
response = llm_with_tools.invoke(messages)
messages.append(response)

if response.tool_calls:

    print("RUNNING TOOLS...")
    tool_messages = execute_tool_calls(response.tool_calls)

    # Extend the history with tool results before the second invoke.
    # The model needs the full conversation — user message, its own tool_call
    # response, and the ToolMessages — to produce a grounded final answer.
    messages.extend(tool_messages)

    final_response = llm_with_tools.invoke(messages)
    print(f"\nFINAL ANSWER = {final_response.content}")

else:
    print(f"MODEL ANSWERED DIRECTLY = {response.content}")

RUNNING TOOLS...
  [convert_units] → 100.0 km = 62.1371 miles

FINAL ANSWER = 100 km is equal to 62.1371 miles.


## Llamadas paralelas

In [16]:
# Both sub-questions are independent: the model can request convert_units and
# calculate in the same response without waiting for one result to inform the
# other. Whether it does so in one shot depends on the model — qwen2.5:3b
# generally issues both calls together for clearly separable sub-questions.
parallel_question = "How many pounds is 75 kg? Also, what is 2 ** 10?"

print(f"QUESTION = {parallel_question}")
messages = [HumanMessage(content = parallel_question)]
response = llm_with_tools.invoke(messages)
messages.append(response)

print(f"\nTOOLS REQUESTED = {len(response.tool_calls)}")
for call in response.tool_calls:
    print(f"  - {call['name']}: {call['args']}")

QUESTION = How many pounds is 75 kg? Also, what is 2 ** 10?

TOOLS REQUESTED = 2
  - convert_units: {'value': 75, 'from_unit': 'kg', 'to_unit': 'lbs'}
  - calculate: {'expression': '2 ** 10'}


In [17]:
if response.tool_calls:

    # execute_tool_calls handles any number of calls uniformly — a single
    # call and a parallel batch follow the exact same code path.
    print("RUNNING ALL TOOL CALLS...")
    tool_messages = execute_tool_calls(response.tool_calls)
    messages.extend(tool_messages)

    final = llm_with_tools.invoke(messages)
    print(f"\nFINAL ANSWER = {final.content}")

else:
    print(f"MODEL ANSWERED DIRECTLY = {response.content}")

RUNNING ALL TOOL CALLS...
  [convert_units] → 75.0 kg = 165.3465 lbs
  [calculate] → 2 ** 10 = 1024

FINAL ANSWER = 75 kg is approximately 165.3465 lbs. 

2 ** 10 equals 1024.


## Manejo de errores y reintentos

### Errores permanentes: el fallo como información

In [18]:
from functools import wraps
import time
# A valid arithmetic expression exercises the happy path through _safe_eval.
print("VALID EXPRESSION:")
try:
    print(" ", calculate.invoke({"expression": "15 * 7 + 3"}))
except Exception as e:
    print("  Error:", e)

# __import__('os').system('ls') parses to a Call node in the AST.
# _safe_eval rejects Call nodes because they are absent from ALLOWED_OPS,
# so the injection attempt raises ValueError before any OS command executes.
print("\nINVALID EXPRESSION (code injection attempt):")
try:
    print(" ", calculate.invoke({"expression": "__import__('os').system('ls')"}))
except Exception as e:
    print("  Error caught:", e)

VALID EXPRESSION:
  15 * 7 + 3 = 108

INVALID EXPRESSION (code injection attempt):
  Error caught: Invalid expression '__import__('os').system('ls')': Unsupported expression type: Call


### Errores transitorios: reintentos automáticos

In [19]:
def with_retry(max_attempts = 3, delay_seconds = 1.0):

    """Decorator factory that transparently retries a function on any exception.

    Wraps the target function in a loop that re-invokes it up to max_attempts
    times, sleeping delay_seconds between each failed attempt. If every attempt
    raises, the last exception is re-raised so the caller sees the original
    error rather than a silent failure.

    Designed to be composed with @tool: applying @with_retry before @tool means
    the retry logic is invisible to LangChain — the model sees a normal tool,
    and transient failures are resolved before a result (or error) reaches the
    agent loop.

    Args:
        max_attempts:  Total number of tries before giving up (default 3).
        delay_seconds: Seconds to sleep between consecutive failures (default 1).

    Returns:
        A decorator that wraps any callable with the retry behaviour.
    """

    def decorator(func):
        # @wraps preserves __name__, __doc__, and __annotations__ so that @tool,
        # applied on top of @with_retry, reads the correct metadata from the
        # original function rather than from the generic wrapper.
        @wraps(func)
        def wrapper(*args, **kwargs):
            last_error = None
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    last_error = e
                    print(f"  [retry] Attempt {attempt}/{max_attempts} failed: {e}")
                    if attempt < max_attempts:
                        time.sleep(delay_seconds)
            raise last_error
        return wrapper
    return decorator


print("DECORATOR 'with_retry' DEFINED")

DECORATOR 'with_retry' DEFINED


### Herramienta con reintentos automáticos

In [20]:
# A dict is used instead of a plain int so the counter can be mutated inside
# get_weather without a 'global' declaration. In Python, closures capture
# bindings, not values — reassigning an int creates a new local variable,
# while mutating a dict key updates the shared object in place.
_call_counter = {"count": 0}
class WeatherInput(BaseModel):
    city: str = Field(description="The city name to get the current weather for")

In [21]:
# Decorators apply bottom-up: @with_retry wraps the function first, then
# @tool wraps the retry-equipped wrapper. When the model invokes the tool it
# calls the wrapper, which handles retries transparently before the agent
# loop ever sees a success or a final failure.
@tool("get_weather", args_schema = WeatherInput)
@with_retry(max_attempts = 3, delay_seconds = 0.5)
def get_weather(city: str) -> str:

    """Get the current weather for a city.

    Simulates an unreliable external API by failing the first two calls with a
    ConnectionError, then succeeding on the third. The failure sequence is
    deterministic: _call_counter tracks invocations across calls and resets on
    success, so the pattern repeats if the tool is invoked again.

    Args:
        city: The name of the city to retrieve weather data for.

    Returns:
        A weather summary string on success. On repeated failure, @with_retry
        re-raises the last ConnectionError after exhausting all attempts.
    """

    _call_counter["count"] += 1
    count = _call_counter["count"]

    # Fail the first two calls to simulate transient network errors.
    if count <= 2:
        raise ConnectionError(f"Weather API timeout on attempt {count}")

    # Reset so the demo sequence repeats if the cell is run again.
    _call_counter["count"] = 0
    return f"Weather in {city}: 22°C, partly cloudy, humidity 65%"

In [22]:
print("CALLING get_weather (will fail twice before succeeding)...\n")
result = get_weather.invoke({"city": "Bogotá"})
print(f"\nRESULT = {result}")

CALLING get_weather (will fail twice before succeeding)...

  [retry] Attempt 1/3 failed: Weather API timeout on attempt 1
  [retry] Attempt 2/3 failed: Weather API timeout on attempt 2

RESULT = Weather in Bogotá: 22°C, partly cloudy, humidity 65%


## Agente completo con ciclo de herramientas

In [23]:
def run_system(user_input: str, max_iterations: int = 5) -> str:

    """Run a tool-calling loop until the model produces a plain-text answer.

    Each iteration invokes llm_with_tools with the full message history. If the
    model responds with one or more tool_calls, execute_tool_calls runs them and
    appends the ToolMessage results to the history before the next iteration.
    When the model responds with content and no tool_calls the loop exits and
    returns that content as the final answer.

    The message history grows with each tool-calling iteration:
        [HumanMessage]
        → [AIMessage(tool_calls=[...])]
        → [ToolMessage, ToolMessage, ...]
        → [AIMessage(content="...")]

    max_iterations guards against infinite loops: if the model never stops
    requesting tools the function returns a sentinel string instead of running
    forever.

    Args:
        user_input:     The user's question or instruction.
        max_iterations: Maximum number of LLM invocations before aborting.
                        Default of 5 is generous for two-tool queries.

    Returns:
        The model's final text answer, or a sentinel string if the iteration
        budget is exhausted without a plain-text response.
    """

    messages = [HumanMessage(content = user_input)]
    print(f"USER: {user_input}\n")

    for iteration in range(1, max_iterations + 1):
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        # No tool_calls means the model has enough context to answer directly.
        if not response.tool_calls:
            print(f"SYSTEM: {response.content}")
            return response.content

        print(f"[iter {iteration}] Tools requested: {[c['name'] for c in response.tool_calls]}")
        tool_messages = execute_tool_calls(response.tool_calls)
        messages.extend(tool_messages)

    return "Max iterations reached without a final answer."


print("SYSTEM DEFINED")

SYSTEM DEFINED


### Ejecución de una consulta compuesta

In [24]:
run_system(
    "How many feet are 100 meters? Also, what is 144 / 12?"
)

USER: How many feet are 100 meters? Also, what is 144 / 12?

[iter 1] Tools requested: ['convert_units', 'calculate']
  [convert_units] → 100.0 meters = 328.0840 feet
  [calculate] → 144 / 12 = 12.0
SYSTEM: 100 meters is approximately 328.084 feet. For the second calculation, 144 divided by 12 equals 12.0.


'100 meters is approximately 328.084 feet. For the second calculation, 144 divided by 12 equals 12.0.'